In [1]:
!uv add gitsource

Resolved 129 packages in 0.95ms
Checked 125 packages in 2ms


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [7]:
question = "How does the agentic loop keep calling the model until it stops?"

In [8]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields= ["filename"]
)

index.fit(documents)

In [9]:
search_results = index.search(
    question,
    num_results=5
)

In [10]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = OpenAI()

In [11]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['content'])

    return '\n\n'.join(lines).strip()

In [12]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [13]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''

In [14]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [15]:
def rag(question,search_index):
    search_results = search_index.search(question,num_results=5)
    prompt = build_prompt(question, search_results)
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input= [
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt}
        ],
    )

    return response

    

In [16]:
response = rag(question, index)
response.usage.input_tokens

7035

In [17]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [18]:
len(chunks)

295

In [19]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields= ["filename"]
)

In [20]:
chunk_index.fit(chunks)

In [21]:
chunk_response = rag(question,chunk_index)

In [22]:
chunk_response.usage.input_tokens

2220

In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, RunnerCallback

In [24]:
instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

In [25]:
def search(query):
    """Search the course lessons for chunks relevant to the query.

    Args:
        query: The search query describing what to look for.
    Returns:
        A list of the most relevant lesson chunks.
    """
    
    return chunk_index.search(
        query,
        num_results=5,)

In [26]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:


runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=None,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [29]:
class SearchCounter(RunnerCallback):
    def __init__(self):
        self.count = 0

    def on_function_call(self, function_call, result):
        if function_call.name == "search":
            self.count += 1

    def on_message(self, message):
        pass

    def on_reasoning(self, reasoning):
        pass

    def on_response(self, response):
        pass


In [30]:
counter = SearchCounter()
runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=counter,
)

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG difference retrieval reasoning tool use"}', call_id='call_oYmxxlunQxsSS34WPmmhRjEI', name='search', type='function_call', id='fc_03577012c629e066006a37fd8021a8819394403496239b4f0e', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop steps observe think act memory planning course"}', call_id='call_TMFV921XVPhqFkYslbCda4oU', name='search', type='function_call', id='fc_03577012c629e066006a37fd8021c0819380e5bf1ec749d05b', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"p

In [31]:
counter.count

3